## Carga de librerias y de la base de datos

In [1]:
import pandas as pd #permite trabajar con excel
import numpy as np #para trabajar con matematicas
import random #generar numeros aleatorios
import copy

In [2]:
#cargamos los datos de consumos REALES
datosConsumoReal = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/ConsumoReal(MEDICAMENTOS)2022-2024.xlsx')

datosConsumoReal.head()
datosConsumoReal.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 390 entries, 0 to 389
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Código         390 non-null    object
 1   Desripción     390 non-null    object
 2   Concentración  390 non-null    object
 3   2022           390 non-null    int64 
 4   2023           390 non-null    int64 
 5   2024           390 non-null    int64 
dtypes: int64(3), object(3)
memory usage: 18.4+ KB


In [3]:
#cargamos los datos de PRECIOS de diferentes fuentes
datosPrecios = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/Precios(MEDICAMENTOS)2022-2024Completo.xlsx')
datosPrecios.head()
datosPrecios.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 390 entries, 0 to 389
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Código                    390 non-null    object 
 1   Desripción                390 non-null    object 
 2   Concentración             390 non-null    object 
 3   Costo2022 (SU)            390 non-null    int64  
 4   Costo2023 (SU)            390 non-null    float64
 5   Costo2024 (SU)            390 non-null    float64
 6   Precio LINAME Junio 2022  381 non-null    float64
 7   Precio LINAME Sep 2023    382 non-null    float64
 8   Precio LINAME Junio 2024  390 non-null    float64
 9   Precio LINAME Nov. 2024   382 non-null    float64
 10  Precio LINAME Abril 2025  382 non-null    float64
 11  Precio LINAME Junio 2025  382 non-null    float64
dtypes: float64(8), int64(1), object(3)
memory usage: 36.7+ KB


In [4]:
#confirmamos que la lista de cantidades con la lista de precios esten ordenadas
#cada medicamento corresponde a su precio
print(datosConsumoReal.loc[335])
print(datosPrecios.loc[335])


Código                                          V-08-03
Desripción                             CONTRASTE IODADO
Concentración    Según disponibilidad (100 ml o 200 ml)
2022                                                 19
2023                                                  0
2024                                                  0
Name: 335, dtype: object
Código                                                     V-08-03
Desripción                                        CONTRASTE IODADO
Concentración               Según disponibilidad (100 ml o 200 ml)
Costo2022 (SU)                                                   0
Costo2023 (SU)                                                 0.0
Costo2024 (SU)                                                 0.0
Precio LINAME Junio 2022                                    312.26
Precio LINAME Sep 2023                                      312.26
Precio LINAME Junio 2024                                    312.26
Precio LINAME Nov. 2024             

In [5]:
#cargo los datos del POA 2024
datosPoa = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/POA(IMEDICAMENTOS) 2024.xlsx')
datosPoa.head()
datosPoa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 390 entries, 0 to 389
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Codigo               390 non-null    object 
 1   Detalle              390 non-null    object 
 2   Unidad               349 non-null    object 
 3   Cant. Inicial        346 non-null    float64
 4   Cant. Entrada        346 non-null    float64
 5   Imp. Entrada         346 non-null    float64
 6   Cant. Salida         346 non-null    float64
 7   CONSUMO ANUAL 2023   346 non-null    float64
 8   Imp. Salida          346 non-null    float64
 9   SALDO  FINAL 2023    346 non-null    float64
 10  Imp/Unit Saldo       346 non-null    float64
 11  Importe Total Saldo  346 non-null    float64
 12  CPM                  346 non-null    float64
 13  CATIDAD A PEDIR      390 non-null    float64
 14  COSTO UNITARIO       390 non-null    float64
 15  TOTAL COSTO          390 non-null    flo

In [6]:
#cargo los datos inventario
datosInventario = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/Inventarios.xlsx')
datosInventario.head()
datosInventario.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 390 entries, 0 to 389
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Código  390 non-null    object
 1   2021    390 non-null    int64 
 2   2022    390 non-null    int64 
 3   2023    390 non-null    int64 
 4   2024    390 non-null    int64 
 5   2025    390 non-null    int64 
dtypes: int64(5), object(1)
memory usage: 18.4+ KB


##PARAMETROS




1. TP: techo presupuestario disponible en el periodo t
2. n: número total de medicamentos i
3. PRA_i^t: precio referencial AGEMED del medicamento i en el periodo t
4. PRE_i^t: precio referencial externo del medicamento i en el periodo t
5. z_i^t: consumo real del medicamento i en el periodo t
6. y_i^t: tasa de crecimiento/decrecimiento del medicamento i en el periodo t
7. S_i^t: saldo (stock) del medicamento i en el periodo t
8. CPM_i: consumo promedio mensual del medicamento i
9. LSMED_i: límite superior de meses de existencia permitidos para el medicamento i



In [58]:
# 1Para el TP sumamos todo los registros de "TOTAL COSTO" de datosPOA
TP = datosPoa['TOTAL COSTO'].sum()  #*2.1
print("Techo presupuestario TP = ",np.round(TP,2))
# 2 para N obtenemos la cantidad de variables en datosConsumoReal
N = datosConsumoReal.shape[0]
print(N)


Techo presupuestario TP =  2283910.22
390


In [8]:
#3. para el PRA obtenemos el precio LINAME 2024 de datosPrecio
#redondeamos a 2 decimales
PRA = datosPrecios['Precio LINAME Junio 2024'] ### preguntar QUE PRECIOS corresponden ###
PRA = np.round(PRA,2)
#print(PRA)
#observacion: hay medicamentos que no tienen precio en LINAME (solucion reemplace del costo del SU)
#verificacion
print(np.isnan(PRA).any())

False


In [9]:
#4. Para el PRE tomamos los valores PRA y le sumamos un 25% a su precio
PRE = PRA * 1.35
PRE = np.round(PRE,2)
#print(PRE)
#Aqui se puede cargar la base de datos con PRE reales!
print(np.isnan(PRE).any())

False


In [10]:
#5. para z sacamos el consumo REAL de las gestiones que necesitamos
#z convertido en array
z = np.array([datosConsumoReal[2022].values, datosConsumoReal[2023].values, datosConsumoReal[2024].values])
print(z)

[[  976 60271  2992 ...     0     0     0]
 [  980 52020  1577 ...     0     0     0]
 [  928 52824  2246 ...     0     0     0]]


In [11]:
#6. realizamos el calculo de la tasa tomando en cuenta
#cantidades gestion 2022 y gestion 2023 (usamos indices para indicar la gestion)
AU = (z[1]-z[0])/z[0]
#Existe problema con la division entre cero:
#infinito = 5/0 (division de cualquier numero entre cero)
#nan = 0/0     (division de cero entre cero)
print(AU[350:375])

#para el caso de valores nan (Not a number) asumimos que no existe crecimiento ni decrecimiento (0.0)
### PREGUNTAR ESTE CASO EN PARTICULAR ###
AU = np.where(np.isnan(AU), 0, AU)
#para el caso especial INFINITO asumimos crecimiento del 100%
#limpiamos valores inf (infinitos) y remplazamos por 1.0
AU = np.where(~np.isfinite(AU), 0, AU)
#verificamos
print(AU[350:375])

#finalmente obtenemos y_i
y = (1+AU)
print(y[350:375])


[-1.         -1.         27.         -0.83333333  1.         -1.
  1.         -1.         -1.          2.12380952         inf         inf
         inf         inf         inf         inf         inf         inf
         inf         nan         nan         nan         nan         nan
         nan]
[-1.         -1.         27.         -0.83333333  1.         -1.
  1.         -1.         -1.          2.12380952  0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.        ]
[ 0.          0.         28.          0.16666667  2.          0.
  2.          0.          0.          3.12380952  1.          1.
  1.          1.          1.          1.          1.          1.
  1.          1.          1.          1.          1.          1.
  1.        ]


/tmp/ipython-input-940264226.py:3: RuntimeWarning: divide by zero encountered in divide
  AU = (z[1]-z[0])/z[0]
/tmp/ipython-input-940264226.py:3: RuntimeWarning: invalid value encountered in divide
  AU = (z[1]-z[0])/z[0]


In [12]:
#7. Si : Saldo total del sku i en el periodo t
S = np.array([datosInventario[2022].values, datosInventario[2023].values, datosInventario[2024].values])
print(S)

[[  259 17098  1037 ...   130     0    11]
 [   79  6378   960 ...     0     0    11]
 [  114 17574   214 ...     0     0     0]]


In [13]:
#	8. CPMi	: Consumo promedio mensual del sku i
# calculando sobre el último año de consumo (2024) dividido entre 12
CPM = z[2] / 12
#print(CPM)
#9. LSMEDi	: Límite superior de meses de existencia disponibles del SKU i
# Este es un valor de gestión, 12 meses para esta prueba
LSMED = 8

In [14]:
#MED
MED = np.divide(S[2], CPM, out=np.zeros_like(S[2], dtype=float), where=CPM!=0)

In [15]:
# Definimos la variable P_i^t (Binaria)
#P es variable de desicion o es parámetro?
P = (MED < LSMED).astype(int)
print(P)

[1 1 1 0 1 0 0 1 1 1 1 0 1 1 0 1 1 1 1 0 1 1 0 1 0 1 1 1 1 1 1 1 1 1 1 1 0
 0 1 1 1 0 1 0 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1
 1 1 1 0 1 1 1 1 1 1 0 1 1 0 1 1 1 1 1 1 1 0 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1
 1 0 1 1 1 1 1 1 1 1 1 1 0 0 1 1 1 0 1 0 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0
 1 1 1 1 1 1 1 1 0 1 1 1 1 1 0 1 0 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1
 1 1 0 1 0 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 0 0 1 1
 1 1 1 0 1 1 1 1 1 0 1 1 1 1 0 1 0 0 0 1 0 1 0 1 1 1 1 1 1 1 1 1 0 1 1 1 1
 1 0 1 1 1 1 1 1 1 0 1 1 1 1 0 1 1 1 1 1 0 1 1 0 1 1 1 0 1 0 1 1 1 1 0 1 1
 1 1 1 1 1 1 1 1 0 1 1 1 1 1 0 0 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0
 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1
 1 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]


## Cálculos necesarios:


In [18]:
# demanda proyectada
d = np.ceil(z[1]*y).astype(int) #redondeado al entero superior

# requerimiento neto de compra ecuacion 15
S_prev = S[1]
r = np.maximum(0, d - S_prev)
r = r.astype(int)

# requerimiento ajustado por P
r = (P * r).astype(int)

## Funciones
ya tenemos la capacidad de referencia ahora pasamo a generar la población inicial utilizando las funciones ya definidas

In [33]:
import numpy as np

# Función objetivo
def calcular_fitness(wi, xi, PRA, PRE):

    # Convertir entradas a arreglos numpy con tipos numéricos seguros
    wi  = np.asarray(wi, dtype=float)
    xi  = np.asarray(xi, dtype=float)
    PRA = np.asarray(PRA, dtype=float)
    PRE = np.asarray(PRE, dtype=float)

    # Costos por tipo de compra
    costo_planificado = float(np.sum(wi * PRA))  # sum_i wi_i * PRA_i
    costo_extra       = float(np.sum(xi * PRE))  # sum_i xi_i * PRE_i

    # Costo total (CT)
    CT = costo_planificado + costo_extra

    return CT, costo_planificado, costo_extra


def generar_poblacion_inicial(pop_size, r, TP, PRA, seed=None, max_intentos=200):
    """
    Población: pop -> shape (pop_size, 2, n)
    Individuo: ind -> shape (2, n) donde ind[0]=w, ind[1]=x

    Restricciones:
      (17) w + x = r
      (14) sum_i w_i * PRA_i <= TP
    """
    rng = np.random.default_rng(seed)

    r = np.asarray(r, dtype=int)
    PRA = np.asarray(PRA, dtype=float)
    TP = float(TP)

    n = r.size
    pop = np.zeros((pop_size, 2, n), dtype=int)

    # Para recortar costo rápido: primero los más caros
    idx_caros = np.argsort(-PRA)

    for k in range(pop_size):
        mejor_w = None
        mejor_costo = -1.0  # para aprovechar presupuesto sin pasarse

        for _ in range(max_intentos):
            # 1) w aleatorio en [0, r]
            w = rng.integers(0, r + 1, size=n, dtype=int)

            # 2) Ajuste para cumplir presupuesto (greedy recortando ítems caros)
            costo = float(np.sum(w * PRA))
            if costo > TP:
                exceso = costo - TP
                for i in idx_caros:
                    if exceso <= 0:
                        break
                    if w[i] == 0 or PRA[i] == 0:
                        continue
                    bajar = int(np.ceil(exceso / PRA[i]))
                    bajar = min(bajar, w[i])
                    w[i] -= bajar
                    exceso -= bajar * PRA[i]

            # 3) Verificación
            costo = float(np.sum(w * PRA))
            if costo <= TP and np.all((0 <= w) & (w <= r)):
                if costo > mejor_costo:
                    mejor_costo = costo
                    mejor_w = w.copy()

        if mejor_w is None:
            # Fallback seguro: todo extra (w=0)
            mejor_w = np.zeros(n, dtype=int)

        x = r - mejor_w  # (17) exacta

        pop[k, 0, :] = mejor_w
        pop[k, 1, :] = x

    return pop

def es_factible(ind, r, TP, PRA):
    w = ind[0]
    x = ind[1]

    # (17)
    ok17 = np.all(w + x == r)

    # no negatividad
    okNN = np.all(w >= 0) and np.all(x >= 0)

    # (14)
    costo_w = float(np.sum(w * PRA))
    ok14 = costo_w <= float(TP) + 1e-9

    return ok17 and okNN and ok14, costo_w

def reparar(ind, r, TP, PRA):
    w = ind[0].copy()
    r = np.asarray(r, dtype=int)
    PRA = np.asarray(PRA, dtype=float)
    TP = float(TP)

    # 1) asegurar enteros y límites
    w = np.asarray(w, dtype=int)
    w = np.clip(w, 0, r)

    # 2) recorte por presupuesto si es necesario
    costo = float(np.sum(w * PRA))
    if costo > TP:
        exceso = costo - TP
        idx_caros = np.argsort(-PRA)  # caros primero
        for i in idx_caros:
            if exceso <= 0:
                break
            if w[i] == 0 or PRA[i] == 0:
                continue
            bajar = int(np.ceil(exceso / PRA[i]))
            bajar = min(bajar, w[i])
            w[i] -= bajar
            exceso -= bajar * PRA[i]

    # 3) reconstruir x para cumplir (17)
    x = r - w

    # 4) devolver individuo reparado
    ind_rep = np.vstack([w, x]).astype(int)
    return ind_rep




In [34]:
pop = generar_poblacion_inicial(pop_size=10, r=r, TP=TP, PRA=PRA, seed=123)

# Individuo k:
ind = pop[0]        # shape (2, n)
w = ind[0]
x = ind[1]

fitness, costo_planificado, costo_extra = calcular_fitness(w, x, PRA, PRE)
print(costo_planificado, costo_extra)
print(np.shape(pop))
print(np.shape(pop[1]))

print(es_factible(ind, r, TP, PRA))

2283907.32 3731803.77
(10, 2, 390)
(2, 390)
(True, 2283907.32)


In [35]:
#FUNCIONES BWO

import numpy as np

def procreacion(parent1, parent2, r, TP, PRA, PRE, CR,
               n_hijos_objetivo=None, rng=None, max_intentos=20000):


    if rng is None:
        rng = np.random.default_rng()

    # Asegurar tipos
    r   = np.asarray(r, dtype=int)
    PRA = np.asarray(PRA, dtype=float)
    PRE = np.asarray(PRE, dtype=float)
    TP  = float(TP)

    n = r.size

    # Cantidad de hijos a intentar generar (antes de canibalismo)
    if n_hijos_objetivo is None:
        n_hijos_objetivo = n  # por defecto, 1 hijo por gen (puedes cambiarlo)
    n_hijos_objetivo = int(n_hijos_objetivo)
    n_hijos_objetivo = max(2, n_hijos_objetivo)

    # Extraer w de los padres (solo cruzamos w)
    w1 = np.asarray(parent1[0], dtype=int)
    w2 = np.asarray(parent2[0], dtype=int)

    children = []
    fit_children = []
    intentos = 0

    while len(children) < n_hijos_objetivo and intentos < max_intentos:
        intentos += 1

        # 1) Elegir n/2 índices únicos
        k = n // 2
        if k == 0:
            idxs = np.array([], dtype=int)
        else:
            idxs = rng.choice(n, size=k, replace=False)

        # 2) Crear hijos copiando a los padres (solo w)
        w_child1 = w1.copy()
        w_child2 = w2.copy()

        # 3) Cruce solo afecta a n/2 genes
        #    Child1[i] = round(alpha*P1[i] + (1-alpha)*P2[i]) en idxs
        alfas = rng.random(size=idxs.size)  # un alfa por gen cruzado
        for j, i in enumerate(idxs):
            alfa = alfas[j]
            w_child1[i] = int(np.rint(alfa * w1[i] + (1.0 - alfa) * w2[i]))
            w_child2[i] = int(np.rint(alfa * w2[i] + (1.0 - alfa) * w1[i]))

        # 4) Limitar w para asegurar x>=0 con w+x=r
        w_child1 = np.clip(w_child1, 0, r)
        w_child2 = np.clip(w_child2, 0, r)

        # 5) Reconstruir x para cumplir (17)
        x_child1 = (r - w_child1).astype(int)
        x_child2 = (r - w_child2).astype(int)

        ind1 = np.vstack([w_child1, x_child1]).astype(int)
        ind2 = np.vstack([w_child2, x_child2]).astype(int)

        # 6) Verificar factibilidad y reparar si no cumple (14) o algo
        fact1, _ = es_factible(ind1, r, TP, PRA)
        if not fact1:
            ind1 = reparar(ind1, r, TP, PRA)
            fact1, _ = es_factible(ind1, r, TP, PRA)

        if fact1:
            CT1, _, _ = calcular_fitness(ind1[0], ind1[1], PRA, PRE)
            children.append(ind1)
            fit_children.append(CT1)

        if len(children) < n_hijos_objetivo:
            fact2, _ = es_factible(ind2, r, TP, PRA)
            if not fact2:
                ind2 = reparar(ind2, r, TP, PRA)
                fact2, _ = es_factible(ind2, r, TP, PRA)

            if fact2:
                CT2, _, _ = calcular_fitness(ind2[0], ind2[1], PRA, PRE)
                children.append(ind2)
                fit_children.append(CT2)

    if len(children) == 0:
        return np.empty((0, 2, n), dtype=int), np.array([], dtype=float)

    children = np.asarray(children, dtype=int)            # (k,2,n)
    fit_children = np.asarray(fit_children, dtype=float)  # (k,)

    # 7) Ordenar por fitness (min)
    idx = np.argsort(fit_children)
    children = children[idx]
    fit_children = fit_children[idx]

    # 8) Canibalismo: mantener una fracción CR de los mejores
    ns = int(len(children) * CR)
    ns = max(1, ns)

    return children[:ns], fit_children[:ns]


import numpy as np

def mutacion_gaussiana(pop1, PM, r, TP, PRA, PRE,
                       sigma=7, frac_genes=0.10, rng=None, max_intentos=20000):

    if rng is None:
        rng = np.random.default_rng()

    r   = np.asarray(r, dtype=int)
    PRA = np.asarray(PRA, dtype=float)
    PRE = np.asarray(PRE, dtype=float)
    TP  = float(TP)

    pop_size = len(pop1)
    n = r.size

    # número de mutantes a generar
    nm = int(pop_size * PM)
    nm = max(1, nm)

    # número de genes a mutar por individuo
    k = int(np.round(frac_genes * n))
    k = max(1, k)

    pop3 = []
    fit_mutation = []

    intentos = 0
    while len(pop3) < nm and intentos < max_intentos:
        intentos += 1

        # Elegir individuo aleatorio
        ind = pop1[rng.integers(0, pop_size)]
        w_mut = np.asarray(ind[0], dtype=int).copy()

        # Mutar k genes
        idxs = rng.choice(n, size=k, replace=False)
        deltas = np.rint(rng.normal(loc=0.0, scale=sigma, size=k)).astype(int)
        for j, i in enumerate(idxs):
            w_mut[i] = w_mut[i] + int(deltas[j])

        # Limitar para que x no sea negativo con (17)
        w_mut = np.clip(w_mut, 0, r)

        # Reconstruir x para cumplir (17)
        x_mut = (r - w_mut).astype(int)

        ind_mut = np.vstack([w_mut, x_mut]).astype(int)

        # Verificar factibilidad y reparar si hace falta
        fact, _ = es_factible(ind_mut, r, TP, PRA)
        if not fact:
            ind_mut = reparar(ind_mut, r, TP, PRA)
            fact, _ = es_factible(ind_mut, r, TP, PRA)

        if not fact:
            continue

        # Fitness (FO)
        CT, _, _ = calcular_fitness(ind_mut[0], ind_mut[1], PRA, PRE)

        pop3.append(ind_mut)
        fit_mutation.append(CT)

    if len(pop3) == 0:
        return np.empty((0, 2, n), dtype=int), np.array([], dtype=float)

    return np.asarray(pop3, dtype=int), np.asarray(fit_mutation, dtype=float)


## MAIN

In [59]:
# PARAMETROS BWO
PR = 0.8    # Procreation rate (fracción de población usada como padres)
CR = 0.4    # Fracción de hijos que sobreviven tras canibalismo (según tu interpretación)
PM = 0.6    # Tasa de mutación (fracción de individuos a mutar)
N_Pop = 100 # Número de widows iniciales
N_Iter = 100  # Iteraciones

nr = int(np.round(PR * N_Pop))
nr = max(1, nr)

PRA = np.asarray(PRA, dtype=float)
PRE = np.asarray(PRE, dtype=float)

# Initialize rng for use in the main loop
semilla=None
rng = np.random.default_rng(seed=semilla) # Added a seed for reproducibility

# GENERAR POBLACION INICIAL (FACTIBLE)
Pop = generar_poblacion_inicial(pop_size=N_Pop, r=r, TP=TP, PRA=PRA, seed=semilla)

# CALCULAR FITNESS INICIAL (FO pura)
Fit_pop = []
for individuo in Pop:
    CT, CP, CE = calcular_fitness(individuo[0], individuo[1], PRA, PRE)
    Fit_pop.append(CT)

Fit_pop = np.asarray(Fit_pop, dtype=float)

# Ordenar por fitness (menor CT mejor)
sorted_indices = np.argsort(Fit_pop)
Fit_pop = Fit_pop[sorted_indices]
Pop = Pop[sorted_indices]

# Guardar historial del mejor fitness y la mejor solucion
fits = []
wis = []

for iter in range(N_Iter):
    # Mostrar estado de la iteracion
    print(f"--------Iteración: {iter+1}/{N_Iter} - Mejor Fitness: {Fit_pop[0]:.2f}")

    # 1) SELECCIÓN DE PARTICIPANTES (tomar los mejores nr)
    Pop1 = copy.deepcopy(Pop[:nr])  # Pop: ndarray (N_Pop, 2, n)

    # Si por alguna razón viene como ndarray y quieres trabajar con listas:
    if isinstance(Pop1, np.ndarray):
        Pop1 = Pop1.tolist()        # cada individuo queda como lista [w, x] (cada uno array)

    # Pool para acumular hijos de TODAS las reproducciones de esta iteración
    ChildrenPool = []
    FitChildrenPool = []

    debiles = set()

    # nr REPRODUCCIONES
    for _ in range(nr):
        # elegir 2 padres del pool de tamaño nr
        idx1, idx2 = rng.choice(nr, size=2, replace=False)
        Parent1 = Pop1[idx1]   # Parent1 = [w, x]
        Parent2 = Pop1[idx2]   # Parent2 = [w, x]

        #-------------------------------------------------------------------------------------
        # PROCREACIÓN (usa r, TP, PRA, PRE, CR y tu rng; ya verifica y repara internamente)
        Pop2, Fit_children = procreacion(parent1=np.asarray(Parent1, dtype=int), parent2=np.asarray(Parent2, dtype=int), r=r, TP=TP, PRA=PRA, PRE=PRE,
            CR=CR,
            n_hijos_objetivo=8,
            rng=rng)

        # Asegurar tipos para acumular como listas (mismo estilo que tu código)
        if isinstance(Pop2, np.ndarray):
            Pop2 = Pop2.tolist()  # cada hijo queda como matriz 2xn en lista
        if isinstance(Fit_children, np.ndarray):
            Fit_children = Fit_children.tolist()

        # Acumular sobrevivientes
        ChildrenPool.extend(Pop2)
        FitChildrenPool.extend(Fit_children)

        # Guardar al "padre más débil" para canibalismo de padres después
        f1, _, _ = calcular_fitness(np.asarray(Parent1[0]), np.asarray(Parent1[1]), PRA, PRE)
        f2, _, _ = calcular_fitness(np.asarray(Parent2[0]), np.asarray(Parent2[1]), PRA, PRE)
        debiles.add(idx1 if f1 > f2 else idx2)

    # CANIBALISMO DE PADRES (se eliminan de Pop y Fit_pop)
    for idx in sorted(debiles, reverse=True):
        Pop = np.delete(Pop, idx, axis=0)
        Fit_pop = np.delete(Fit_pop, idx, axis=0)

    # Pools listos
    ChildrenPool = np.array(ChildrenPool, dtype=int)
    FitChildrenPool = np.array(FitChildrenPool, dtype=float)

    #-------------------------------------------------------------------------------------
    # Mutación (gaussiana) usando la función actualizada
    sigma = 50.0 * (1.1 - iter / N_Iter)     # Sigma decreciente
    sigma = max(1.0, sigma)

    Pop1_arr = np.asarray(Pop1, dtype=int) if not isinstance(Pop1, np.ndarray) else Pop1

    Pop3, Fit_mutation = mutacion_gaussiana(pop1=Pop1_arr, PM=PM, r=r, TP=TP, PRA=PRA, PRE=PRE, sigma=sigma, frac_genes=0.20, rng=rng)

    #-------------------------------------------------------------------------------------
    # Actualización de población: combinar población + hijos + mutados
    Pop = np.concatenate((Pop, ChildrenPool, Pop3), axis=0)

    # Fitness: actuales + hijos + mutados
    Fit_pop = np.concatenate((Fit_pop, FitChildrenPool, Fit_mutation), axis=0).astype(float)

    # Ordenar por fitness (menor mejor)
    sorted_indices = np.argsort(Fit_pop)
    Fit_pop = Fit_pop[sorted_indices]
    Pop = Pop[sorted_indices]

    # Recortar a tamaño fijo N_Pop
    Pop = Pop[:N_Pop]
    Fit_pop = Fit_pop[:N_Pop]

    # Guardar el mejor fitness y la mejor widow de esta iteración
    fits.append(float(Fit_pop[0]))
    wis.append(Pop[0].copy())   # Pop[0] es ndarray (2,n); copy() basta

# REPORTE FINAL
print("__________________________________________________________")
print(f"MEJOR COSTO ENCONTRADO (CT): {fits[-1]:.2f} Bs")
print("MEJOR WIDOW:")

w_best = np.asarray(wis[-1][0], dtype=int)
x_best = np.asarray(wis[-1][1], dtype=int)

print(f"  w (planificadas): {w_best[0:10]}")
print(f"  x (extras):       {x_best[0:10]}")
print(f"  r (requerimiento):{np.asarray(r[0:10], dtype=int)}")
print(f"  d (demanda proj):  {np.asarray(d[0:10], dtype=int)}")

# Detalle de costos de la mejor solución
# ------------------------------------------------------------
w_mejor = w_best.astype(float)
x_mejor = x_best.astype(float)

costo_planificado = float(np.sum(w_mejor * PRA))   # CP = sum_i w_i * PRA_i
costo_extra       = float(np.sum(x_mejor * PRE))   # CE = sum_i x_i * PRE_i
costo_total       = costo_planificado + costo_extra  # CT

print("\nDetalle de costos:")
print(f"  Costo planificado (CP):          {costo_planificado:.2f} Bs")
print(f"  Costo extra (CE):                {costo_extra:.2f} Bs")
print(f"  Costo total compra (CT=CP+CE):   {costo_total:.2f} Bs")
print(f"  Presupuesto (TP):                {float(TP):.2f} Bs")

# Verificación útil para restricción (14)
print("__________________________________________________________")
print(f"Holgura (TP - Costo Planificado):  {float(TP - costo_planificado):.2f} Bs")


--------Iteración: 1/100 - Mejor Fitness: 6015397.53
--------Iteración: 2/100 - Mejor Fitness: 6015361.13
--------Iteración: 3/100 - Mejor Fitness: 6015361.13
--------Iteración: 4/100 - Mejor Fitness: 6015346.27
--------Iteración: 5/100 - Mejor Fitness: 6015337.74
--------Iteración: 6/100 - Mejor Fitness: 6015305.25
--------Iteración: 7/100 - Mejor Fitness: 6015246.83
--------Iteración: 8/100 - Mejor Fitness: 6015246.83
--------Iteración: 9/100 - Mejor Fitness: 6015240.33
--------Iteración: 10/100 - Mejor Fitness: 6015229.54
--------Iteración: 11/100 - Mejor Fitness: 6015210.70
--------Iteración: 12/100 - Mejor Fitness: 6015200.47
--------Iteración: 13/100 - Mejor Fitness: 6015200.47
--------Iteración: 14/100 - Mejor Fitness: 6015199.61
--------Iteración: 15/100 - Mejor Fitness: 6015186.33
--------Iteración: 16/100 - Mejor Fitness: 6015185.53
--------Iteración: 17/100 - Mejor Fitness: 6015179.22
--------Iteración: 18/100 - Mejor Fitness: 6015175.08
--------Iteración: 19/100 - Mejor Fit

In [69]:
print("proyeccion del medicamento para el 2024 es de " , d[9])
print("saldo del medicamento 2023 es de ", S[1][9])
print("consumo promedio mensual del medicamento es", z[1][9]/12)
print("meses de existencia ", S[1][9]/(z[1][9]/12))

proyeccion del medicamento para el 2024 es de  998
saldo del medicamento 2023 es de  1093
consumo promedio mensual del medicamento es 108.08333333333333
meses de existencia  10.112567463377024


In [73]:
pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 3.8 MB/s eta 0:00:00


In [75]:
#Guardamos resultado en excel:
#base de datos para guardar:
base = datosConsumoReal[["Código", "Desripción"]].copy()

# 3) Agregar columnas (una columna por vector)
base["w_planificadas"] = w_best.astype(int)
base["x_extras"]       = x_best.astype(int)
base["r_requerimiento"]= r
base["d_demanda_proj"] = d

# 4) Guardar a Excel
out_path = "/content/drive/MyDrive/Colab Notebooks/resultados_optimizacion.xlsx"
with pd.ExcelWriter(out_path, engine="xlsxwriter") as writer:
    base.to_excel(writer, sheet_name="Resultados", index=False)

print("Listo, guardado en:", out_path)
print(base.head(10))


Listo, guardado en: /content/drive/MyDrive/Colab Notebooks/resultados_optimizacion.xlsx
    Código                                    Desripción  w_planificadas  \
0  A-02-01              HIDROXIDO DE ALUMINIO Y MAGNESIO             244   
1  A-02-02                                     OMEPRAZOL           29423   
2  A-02-03                                    RANITIDINA               0   
3  A-02-04                                    RANITIDINA               0   
4  A-02-05                                     OMEPRAZOL             898   
5  A-03-01                              ATROPINA SULFATO               0   
6  A-03-02  BUTILBROMURO DE HIOSCINA (BUTILESCOPOLAMINA)               0   
7  A-03-04  BUTILBROMURO DE HIOSCINA (BUTILESCOPOLAMINA)             275   
8  A-03-06                                   DOMPERIDONA            5334   
9  A-03-07                                METOCLOPRAMIDA               0   

   x_extras  r_requerimiento  d_demanda_proj  
0       662              906